# Train Achoose's own wake word ("Hey Achoose" and "Achoose")

This notebook teaches a small wake word model to recognise **"Hey Achoose"** and **"Achoose"**.
It runs on Google's computers (a free graphics card), not on your laptop. At the end you download
one small file (`hey_achoose.onnx`, under 1 MB) and put it in `~/Achoose/wake_words/` on your laptop.

**How it works, in short**
1. A computer voice says "Hey Achoose" and "Achoose" thousands of times, in hundreds of different voices and speeds.
2. It also says words that *sound similar* (so Achoose learns what NOT to wake up for).
3. Those clips get mixed with room echo and background noise, so they sound like a real room.
4. The model learns from all of that, plus about 2,000 hours of ordinary speech, music and noise that must never wake it.

Nothing from your laptop is uploaded. No recording of your own voice is needed.

**Before you start:** click **Runtime → Change runtime type → T4 GPU → Save**.
Then run the cells one by one, from top to bottom (click a cell, press **Shift + Enter**).
Keep this browser tab open while it runs. All together it takes roughly 1 to 1.5 hours.

## Step 0: Check that a graphics card is connected
This only prints information about the graphics card. If it says *no GPU*, change the runtime type (see above) and run it again.

In [ ]:
import subprocess
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
    print("OK: a graphics card is connected.")
except FileNotFoundError:
    print("No GPU! Click Runtime -> Change runtime type -> T4 GPU, then run this cell again.")

## Step 1: Settings
The phrases to learn and how much to practise. You don't need to change anything.
`n_samples` is how many example clips are made, `steps` is how long the model trains.

In [ ]:
PHRASES = ["hey achoose", "achoose"]           # what should wake Achoose
PRONUNCIATIONS = {"achoose": "AH0 CH UW1 Z"}   # "uh-CHOOSE" (the word is not in the English dictionary)
MODEL_NAME = "hey_achoose"                     # the downloaded file will be hey_achoose.onnx
N_SAMPLES = 10000                              # example clips for training
N_SAMPLES_VAL = 2000                           # example clips for testing
STEPS = 40000                                  # training steps

open("/content/settings.py", "w").write(f"PHRASES = {PHRASES!r}\n")  # the helper scripts read this

## Step 2: Install the tools (about 5 minutes)
What this cell does:
- Colab's own Python changes over time, and the training tools need an older one. So this cell sets up a
  **separate Python 3.12** in `/content/venv` (with the small tool `uv`) and installs everything there,
  pinned to exact versions that were tested together. Colab's own Python is not touched.
- Installs **openWakeWord** (the same wake word system Achoose uses on your laptop) with its training parts.
- Installs **piper-sample-generator**, which makes the computer-voice clips, and downloads its voice model
  (about 200 MB, with hundreds of different English speakers).
- Fixes a small problem in the training script (a setting that was always on by mistake).

At the end it prints **Tools installed.** If you see red errors instead, send a screenshot.

In [ ]:
%cd /content
!pip install -q uv
!rm -rf /content/venv && uv venv -q --python 3.12 --python-preference only-managed /content/venv
PIP = "uv pip install -q --python /content/venv/bin/python"
!{PIP} "torch==2.4.1" "torchaudio==2.4.1" "numpy>=2,<2.3" "scipy==1.14.1" "setuptools<70" \
    "piper-tts==1.3.0" webrtcvad "audiomentations==0.33.0" "torch-audiomentations==0.11.0" \
    "speechbrain==0.5.16" "torchmetrics==0.11.4" "torchinfo==1.8.0" "acoustics==0.2.6" "mutagen==1.48.1" \
    "pronouncing==0.3.0" "deep-phonemizer==0.0.19" "datasets==2.21.0" "onnx==1.17.0" "onnxruntime==1.22.1" \
    pyyaml tqdm soundfile librosa

# openWakeWord (training code), at a tested version
!rm -rf oww_repo && git clone -q https://github.com/dscripka/openWakeWord oww_repo
!cd oww_repo && git checkout -q 368c037
!{PIP} --no-deps -e /content/oww_repo
# Fix: the "convert to tflite" option is always on by mistake; we only need the .onnx file
!sed -i 's/if args.convert_to_tflite:/if args.convert_to_tflite is True:/' oww_repo/openwakeword/train.py

# Clip generator and its multi-speaker voice model
!rm -rf piper-sample-generator && git clone -q https://github.com/rhasspy/piper-sample-generator
!cd piper-sample-generator && git checkout -q v3.2.0
!wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt \
    https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt

# Small models openWakeWord uses to turn sound into features, then a quick check that everything loads
!/content/venv/bin/python -c "import openwakeword.utils as u; u.download_models(['hey_jarvis'])"
!/content/venv/bin/python -c "import torch, openwakeword.train, piper; print('GPU for training:', torch.cuda.is_available()); print('Tools installed.')" 2>&1 | grep -v "Warning\|warn\|set_audio_backend\|torchvision"

## Step 3: Connect the clip generator to the training script
The training script was written for an older version of the clip generator. This cell writes a small
"bridge" file that:
- hands the training script the voice model,
- converts the clips to 16,000 Hz (what wake word models listen at),
- teaches it how "Achoose" sounds,
- makes sure "Achoose" itself is never used as a *negative* example (since "Achoose" is part of "Hey Achoose").

In [ ]:
bridge = f"""
import os, sys
from math import gcd
HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
from piper_sample_generator.__main__ import generate_samples as _generate
import pronouncing

MODEL = os.path.join(HERE, "models", "en_US-libritts_r-medium.pt")
WAKE_PHRASES = {set(p.lower() for p in PHRASES)!r}

pronouncing.init_cmu()
for word, phones in {PRONUNCIATIONS!r}.items():
    if not pronouncing.lookup.get(word):
        pronouncing.lookup[word].append(phones)


def generate_samples(text, output_dir, **kwargs):
    kwargs.pop("auto_reduce_batch_size", None)
    if isinstance(text, list) and any(t.lower() not in WAKE_PHRASES for t in text):
        text = [t for t in text if t.lower() not in WAKE_PHRASES]  # never train our own phrase as "wrong"
    _generate(text=text, output_dir=output_dir, model=MODEL, **kwargs)
    to_16k(output_dir)


def to_16k(folder):
    import numpy as np, scipy.io.wavfile, scipy.signal
    for name in os.listdir(folder):
        if not name.endswith(".wav"):
            continue
        path = os.path.join(folder, name)
        rate, audio = scipy.io.wavfile.read(path)
        if rate == 16000:
            continue
        g = gcd(rate, 16000)
        audio = scipy.signal.resample_poly(audio.astype(np.float32), 16000 // g, rate // g)
        scipy.io.wavfile.write(path, 16000, np.clip(audio, -32768, 32767).astype(np.int16))
"""
open("/content/piper-sample-generator/generate_samples.py", "w").write(bridge)
print("Bridge written.")

## Step 4: Listen to how the computer voices say it (1 minute)
Makes 6 short examples and plays them. **Listen:** does it sound like the way you say "Achoose"?
If it sounds clearly wrong, stop here and tell Claude what it sounds like, so the pronunciation can be fixed
before the long run.

In [ ]:
%%writefile /content/preview.py
import sys, shutil
sys.path.insert(0, "/content/piper-sample-generator")
from generate_samples import generate_samples
from settings import PHRASES
shutil.rmtree("/content/preview", ignore_errors=True)
generate_samples(text=PHRASES, output_dir="/content/preview", max_samples=6, batch_size=6,
                 length_scales=[0.9, 1.0, 1.1], noise_scales=[0.98], noise_scale_ws=[0.98])

In [ ]:
import glob
from IPython.display import Audio, display
!cd /content && /content/venv/bin/python preview.py 2>&1 | grep -i "error" || true
for path in sorted(glob.glob("/content/preview/*.wav")):
    display(Audio(path))

## Step 5: Download practice material (about 10 to 15 minutes)
All of it is public research data, downloaded straight to Google's computer (not to your laptop):
- **Room echoes** (MIT): recordings of how sound bounces in real rooms, to make clips sound natural.
- **Background noise** (one part of Google's AudioSet): everyday sounds like traffic, music, dogs, kitchens.
- **About 2,000 hours of ordinary audio, already turned into features** (about 17 GB): speech, music and noise
  that must *never* wake Achoose. This is what keeps false wake-ups low.
- **About 11 hours of test audio**, to measure how often it would wake up by mistake.

In [ ]:
%%writefile /content/get_data.py
import os, numpy as np, scipy.io.wavfile, datasets, subprocess
from pathlib import Path
from tqdm import tqdm
os.chdir("/content")

print("Room echoes...")
os.makedirs("mit_rirs", exist_ok=True)
for row in tqdm(datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)):
    name = row["audio"]["path"].split("/")[-1]
    scipy.io.wavfile.write(os.path.join("mit_rirs", name), 16000, (row["audio"]["array"] * 32767).astype(np.int16))

print("Background noise...")
os.makedirs("audioset", exist_ok=True)
subprocess.run("wget -q -O audioset/bal_train09.tar https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar"
               " && cd audioset && tar -xf bal_train09.tar", shell=True, check=True)
os.makedirs("background", exist_ok=True)
clips = datasets.Dataset.from_dict({"audio": [str(p) for p in Path("audioset/audio").glob("**/*.flac")]})
clips = clips.cast_column("audio", datasets.Audio(sampling_rate=16000))
for row in tqdm(clips):
    name = row["audio"]["path"].split("/")[-1].replace(".flac", ".wav")
    scipy.io.wavfile.write(os.path.join("background", name), 16000, (row["audio"]["array"] * 32767).astype(np.int16))
subprocess.run("rm -rf audioset", shell=True)
print("Rooms:", len(os.listdir("mit_rirs")), " Background clips:", len(os.listdir("background")))

In [ ]:
!cd /content && /content/venv/bin/python get_data.py 2>&1 | grep -v "Warning\|warn"
# Ordinary audio that must never wake it (features, about 17 GB), and the test set
!cd /content && wget -q --show-progress -c https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!cd /content && wget -q --show-progress -c https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy
!ls -la /content/*.npy

## Step 6: Write the training plan
This saves all the settings into one file (`my_model.yaml`) that the training script reads.

In [ ]:
import yaml
config = yaml.safe_load(open("/content/oww_repo/examples/custom_model.yml"))
config.update({
    "model_name": MODEL_NAME,
    "target_phrase": PHRASES,
    "custom_negative_phrases": [],
    "n_samples": N_SAMPLES,
    "n_samples_val": N_SAMPLES_VAL,
    "steps": STEPS,
    "tts_batch_size": 50,
    "piper_sample_generator_path": "/content/piper-sample-generator",
    "output_dir": "/content/my_custom_model",
    "rir_paths": ["/content/mit_rirs"],
    "background_paths": ["/content/background"],
    "background_paths_duplication_rate": [1],
    "false_positive_validation_data_path": "/content/validation_set_features.npy",
    "feature_data_files": {"ACAV100M_sample": "/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy"},
    "batch_n_per_class": {"ACAV100M_sample": 1024, "adversarial_negative": 50, "positive": 50},
    "target_accuracy": 0.7,
    "target_recall": 0.5,
    "target_false_positives_per_hour": 0.2,
})
yaml.safe_dump(config, open("/content/my_model.yaml", "w"))
print("Training plan saved.")

## Step 7: Make the example clips (about 15 to 25 minutes)
The computer voices say "Hey Achoose" and "Achoose" (and similar-sounding words) thousands of times.
If it stops with an error, just run this cell again: it continues where it left off.

In [ ]:
!cd /content && /content/venv/bin/python oww_repo/openwakeword/train.py --training_config my_model.yaml --generate_clips 2>&1 | grep -v "DEBUG"

## Step 8: Make the clips sound like a real room (about 15 to 30 minutes)
Adds room echo, background noise and volume changes to every clip, then turns them into features.

In [ ]:
!cd /content && /content/venv/bin/python oww_repo/openwakeword/train.py --training_config my_model.yaml --augment_clips 2>&1 | grep -v "DEBUG\|Warning\|warn"

## Step 9: Train the model (about 20 to 40 minutes)
The model practises telling "Hey Achoose" / "Achoose" apart from everything else.
At the end it prints its test results: **recall** (how often it hears the phrase) and
**false positives per hour** (how often it would wake up by mistake). Send Claude a screenshot of the last lines.

In [ ]:
!cd /content && /content/venv/bin/python oww_repo/openwakeword/train.py --training_config my_model.yaml --train_model 2>&1 | grep -v "DEBUG"

## Step 10: Download your wake word file
This downloads `hey_achoose.onnx` to your computer (usually into your Downloads folder).
Then, on your laptop, move it into place with:

```
mkdir -p ~/Achoose/wake_words && mv ~/Downloads/hey_achoose.onnx ~/Achoose/wake_words/
```

In [ ]:
import os
from google.colab import files
path = f"/content/my_custom_model/{MODEL_NAME}.onnx"
print(f"{path}: {os.path.getsize(path) / 1024:.0f} KB")
files.download(path)